# VallaVision: detección de vallas como servicio

**El caso de uso.** la plataforma es una plataforma de etiquetado geolocalizado de imágenes: vehículos de relevamiento recorren la
ciudad con una cámara y cada frame llega con su posición GPS (se ve sobreimpresa en las fotos). VallaVision es el
**microservicio de detección** pensado para integrarse a esa plataforma: recibe un frame y devuelve dónde hay vallas
publicitarias, para que la plataforma las etiquete en el mapa.

Este notebook muestra el servicio por dentro, como lo vería la plataforma que lo consume. Airflow, MLflow y MinIO se
muestran en el navegador.

| Pieza | Papel en la integración |
|---|---|
| Gateway REST | la API que la plataforma llama para etiquetar una imagen |
| Redpanda + consumidor | la ingesta de los vehículos: absorbe ráfagas y conserva el orden de cada recorrido |
| `predictions/` en el Data Lake | las etiquetas que consume la plataforma |
| Gate de promoción + rollback | ningún modelo nuevo cambia las etiquetas sin pasar el gate |
| Monitoreo de drift | aviso cuando cambian las condiciones de captura (de noche, otra ciudad, otra cámara) |

> Requiere el stack arriba y con champion (`registrar_modelo` corrido). Los secretos se leen de `.env` y no se imprimen.

In [ ]:
%matplotlib inline
from demo_utils import *

IMAGEN = "042e2f0b-229c-46b6-8236-1e3391cb3682_1647.jpg"   # del test: un video que el modelo no vio
estado()

## 1 · La plataforma pide etiquetar un frame

`POST /v1/detect` con su API key. El gateway no tiene el modelo: autentica, valida y reenvía la imagen por gRPC al
servicio de inferencia. La respuesta trae las cajas y la **versión del modelo** que las produjo: la plataforma puede
saber con qué modelo se generó cada etiqueta.

In [ ]:
r = detectar(IMAGEN)
mostrar(IMAGEN, r)
tabla(r)

Otros recorridos, otras calles (todos del test revisado: videos que el modelo no vio):

In [ ]:
mosaico([
    "62450afd-c5c6-4438-9127-4aa4c21754f3_1671.jpg",
    "042e2f0b-229c-46b6-8236-1e3391cb3682_2927.jpg",
    "91ff0e05-653a-43bf-a09b-c73aa3a37045_164.jpg",
    "8ed4343d-a1d1-4dc5-a4a0-70cef4d52c38_1755.jpg",
])

## 2 · Adentro: gRPC

Entre el gateway y la inferencia el protocolo es gRPC: la imagen viaja como bytes, con contrato tipado
(`proto/detector.proto`), deadlines y cancelación. Es un servicio interno: en la integración, la plataforma solo ve el
gateway. Misma imagen por los dos caminos:

In [ ]:
comparar_protocolos(IMAGEN)

## 3 · Seguridad

Un servicio que se integra a una plataforma no puede aceptar cualquier cosa. Cada fila es un ataque real contra el sistema
que está corriendo: sin clave, clave inválida, un GIF disfrazado de JPEG, una bomba de descompresión, un archivo enorme,
una acción de administrador y una llamada directa al modelo saltando el gateway.

In [ ]:
pruebas_de_seguridad(IMAGEN)

## 4 · Ingesta de los vehículos de relevamiento (Redpanda)

Tres recorridos (videos del test) publican sus frames en el topic `frames`, como lo harían los vehículos en la calle.
El recorrido es la clave: sus frames caen siempre en la misma partición y quedan en orden. Los vehículos producen más
rápido de lo que el modelo procesa en CPU; el broker absorbe la diferencia. El consumidor infiere y arma ventanas de 15 s
por el **instante de captura**, no de llegada.

In [ ]:
normal = transmitir()

In [ ]:
ventanas(normal)

In [ ]:
particiones()

## 5 · Los mismos recorridos, de noche

El mismo recorrido con los frames oscurecidos. Las métricas operativas casi no cambian: el problema no es de latencia,
es de los datos.

In [ ]:
noche = transmitir(degradar="oscurecer")

In [ ]:
ventanas(noche)

## 6 · Drift: ¿cambiaron las condiciones de captura?

En producción no hay etiquetas para medir el mAP: se vigila si las entradas cambian respecto de la referencia (el `val`
revisado). Con el tráfico que acabamos de mandar:

In [ ]:
comparar_brillo(normal, noche)

El recorrido normal (videos que no están en la referencia) no da drift; el de noche sí. Para la plataforma es la señal de
que las etiquetas de ese tráfico pueden ser menos confiables.

El umbral es α = 0,01, no 0,05: el sistema hace cuatro tests por ventana y frames consecutivos se parecen entre sí, así que
con 0,05 habría falsas alarmas. Esto mismo lo hace cada hora el DAG `monitoreo_drift`, sobre todo el tráfico y con los cuatro
tests corregidos por Holm: **siguiente paso, en el navegador: disparar `monitoreo_drift` en Airflow y abrir la alerta en MinIO.**